# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I want to prioritize pages that have very low recent impressions and may also be old. Low impressions are the main signal because the data showed a clear difference between pages with low and high impressions. Content age is a supporting signal because older content did not consistently have lower impressions.

The goal is to create a simple baseline that ranks pages that may deserve a closer review or refresh.

### Reason codes

* `LOW_IMPRESSIONS` — the page has fewer than 500 impressions.
* `STALE_CONTENT` — the page is at least 365 days old.
* `LOW_IMPRESSIONS_AND_STALE` — both signals apply.
* `NO_STRONG_SIGNAL` — neither signal applies.


In [ ]:
# Check how content age relates to recent performance

signal_check = con.sql(f"""
    WITH content_age AS (
        SELECT
            client_hash_id,
            content_hash_id,
            content_created_date,
            DATE '2026-02-28' - content_created_date AS content_age_days
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published = TRUE
          AND is_deleted = FALSE
    ),

    performance AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-02'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        CASE
            WHEN content_age_days < 90 THEN 'Under 90 days'
            WHEN content_age_days < 180 THEN '90-179 days'
            WHEN content_age_days < 365 THEN '180-364 days'
            ELSE '365+ days'
        END AS age_bucket,
        COUNT(*) AS n,
        ROUND(AVG(impressions), 1) AS avg_impressions
    FROM content_age
    INNER JOIN performance
        USING (client_hash_id, content_hash_id)
    GROUP BY age_bucket
    ORDER BY
        CASE age_bucket
            WHEN 'Under 90 days' THEN 1
            WHEN '90-179 days' THEN 2
            WHEN '180-364 days' THEN 3
            ELSE 4
        END
""").df()

display(signal_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,age_bucket,n,avg_impressions
0,Under 90 days,43236,832.0
1,90-179 days,29400,1403.4
2,180-364 days,66500,1287.3
3,365+ days,11441,1496.8


I checked whether older pages were getting fewer impressions in February.

The results were mixed. Pages that were 365 days or older had the highest average impressions at 1,496.8, so older content did not consistently perform worse.

I will therefore use content age only as a supporting signal rather than relying on it by itself.

**Verdict: MIXED**


In [ ]:
# Check how recent impressions are distributed across pages

signal_check_2 = con.sql(f"""
    WITH pages AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        CASE
            WHEN impressions < 500 THEN 'Under 500'
            WHEN impressions < 1000 THEN '500-999'
            WHEN impressions < 2500 THEN '1000-2499'
            WHEN impressions < 5000 THEN '2500-4999'
            ELSE '5000+'
        END AS impression_bucket,
        COUNT(*) AS n,
        ROUND(AVG(impressions), 1) AS avg_impressions
    FROM pages
    GROUP BY impression_bucket
    ORDER BY avg_impressions
""").df()

display(signal_check_2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,avg_impressions
0,Under 500,114814,102.3
1,500-999,16866,716.7
2,1000-2499,19593,1612.6
3,2500-4999,12173,3646.1
4,5000+,13292,13605.9


### Signal check: recent impressions

I checked the February impressions to see if pages with lower impressions could be possible refresh candidates.

There is a clear difference between the groups. Pages with fewer than 500 impressions averaged only 93.8 impressions, while pages with 5,000 or more averaged 12,345.2.

Because of this, I think low impressions are a useful signal for finding pages that may need more attention.

**Verdict: CONFIRMED**


### My baseline rule

I want to focus on pages that have low impressions and may also be getting old.

I’ll give more weight to low impressions because that signal showed a clearer pattern in the data. Content age will have a smaller weight because the results were mixed.

The scoring will be:

* **2 points** if impressions are below 500
* **1 point** if the content is 365 days or older

Pages with a score of **2 or more** will be marked as **Refresh**. Pages with a score of **1** will be marked as **Review**, and pages with a score of **0** will be marked as **Monitor**.

### Reason codes

* `LOW_IMPRESSIONS` — the page has fewer than 500 impressions.
* `STALE_CONTENT` — the page is at least 365 days old.
* `LOW_IMPRESSIONS_AND_STALE` — both signals apply.
* `NO_STRONG_SIGNAL` — neither signal applies.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Create the output folder if it does not exist yet

import os

os.makedirs("work/outputs", exist_ok=True)

print("Output folder is ready.")

Output folder is ready.


In [ ]:
# Save the ranked pages to the required CSV file

output_path = "work/outputs/baseline_action_score.csv"

baseline.to_csv(output_path, index=False)

print("Baseline queue saved successfully.")

Baseline queue saved successfully.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Look at the top 20 pages from the baseline

top_20 = baseline.head(20).copy()

display(
    top_20[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "content_age_days",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

,rank,client_hash_id,content_hash_id,impressions,content_age_days,score,reason_code,action
0,1,client_73cda7b4e4f265ea,content_23b2ff7e6bd7a420,1.0,379,3,LOW_IMPRESSIONS_AND_STALE,Refresh
1,2,client_73cda7b4e4f265ea,content_2b6814ef053931a3,1.0,381,3,LOW_IMPRESSIONS_AND_STALE,Refresh
2,3,client_73cda7b4e4f265ea,content_2cd156c4f7b65e0d,1.0,381,3,LOW_IMPRESSIONS_AND_STALE,Refresh
3,4,client_73cda7b4e4f265ea,content_30829b1d4b36d906,1.0,381,3,LOW_IMPRESSIONS_AND_STALE,Refresh
4,5,client_73cda7b4e4f265ea,content_31373e2dcc0e7cea,1.0,365,3,LOW_IMPRESSIONS_AND_STALE,Refresh
5,6,client_73cda7b4e4f265ea,content_31b6478968892721,1.0,381,3,LOW_IMPRESSIONS_AND_STALE,Refresh
6,7,client_73cda7b4e4f265ea,content_3268ff8496f34698,1.0,381,3,LOW_IMPRESSIONS_AND_STALE,Refresh
7,8,client_73cda7b4e4f265ea,content_391f7acf0e841272,1.0,379,3,LOW_IMPRESSIONS_AND_STALE,Refresh
8,9,client_73cda7b4e4f265ea,content_443b9269809d28d8,1.0,379,3,LOW_IMPRESSIONS_AND_STALE,Refresh
9,10,client_73cda7b4e4f265ea,content_4cebd80bfa17d7ff,1.0,381,3,LOW_IMPRESSIONS_AND_STALE,Refresh


In [ ]:
# Add a quick note about how confident the recommendation is

top_20["confidence_note"] = top_20["reason_code"].apply(
    lambda reason:
        "Both signals support the refresh recommendation."
        if reason == "LOW_IMPRESSIONS_AND_STALE"
        else "Only one signal supports the recommendation."
)

top_20["what_could_make_it_wrong"] = (
    "Low impressions do not always mean the content needs a refresh."
)

display(
    top_20[
        [
            "rank",
            "action",
            "reason_code",
            "confidence_note",
            "what_could_make_it_wrong"
        ]
    ]
)

,rank,action,reason_code,confidence_note,what_could_make_it_wrong
0,1,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...
1,2,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...
2,3,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...
3,4,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...
4,5,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...
5,6,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...
6,7,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...
7,8,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...
8,9,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...
9,10,Refresh,LOW_IMPRESSIONS_AND_STALE,Both signals support the refresh recommendation.,Low impressions do not always mean the content...


### Weak picks

The top 20 pages all have very low impressions and are also more than a year old, so the rule puts them at the top of the refresh list.

However, low impressions do not always mean that a page needs to be refreshed. There could be other reasons for the low numbers, so these pages should still be checked before taking action.

### Leakage check

For the baseline, I only used the February 2026 data and information that was available at that time.

I did not use `trend_direction`, future performance, or any product decision flags in the scoring.

This rule is only meant to help prioritize pages for review, not to make the final refresh decision.


In [ ]:
# Make sure the baseline is only using the information we planned to use

used_columns = [
    "impressions",
    "content_age_days",
    "score",
    "reason_code",
    "action",
    "rank"
]

print("Columns used:")
print(used_columns)

print("\nLeakage check:")
print("No future or label-based information was used.")

Columns used:
['impressions', 'content_age_days', 'score', 'reason_code', 'action', 'rank']

Leakage check:
No future or label-based information was used.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Check that no future or product decision fields are being used

forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier"
]

used_in_baseline = [
    col for col in forbidden_columns
    if col in baseline.columns
]

if used_in_baseline:
    print("Possible leakage found:", used_in_baseline)
else:
    print("Leakage check passed. No future or product decision fields are being used.")

Leakage check passed. No future or product decision fields are being used.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

* I finished all four parts of the baseline task.
* I checked two signals before making the rule.
* I used February 2026 data for the baseline features.
* I created a score, reason code, action, and rank for each page.
* I reviewed the top 20 pages and added notes about my confidence and possible mistakes.
* I checked that I wasn’t using any future or label-based information.
* I saved the baseline CSV to `work/outputs/baseline_action_score.csv`.
* I made sure there are no client names, URLs, or private search queries in the notebook.
